# Titanic

## Part 1: Data Cleaning

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('titanic.csv')

In [3]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df["Age"] = df.groupby(["Sex", "Pclass"])["Age"].transform(lambda series: series.fillna(series.median()))
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Fare"] = df["Fare"].fillna(df["Fare"].median())

df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace([
    "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
    "Rev", "Sir", "Jonkheer", "Dona", "Mme", "Mlle", "Ms"
], "Rare")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)
df.dropna(inplace=True)

In [5]:
df.isna().sum()

Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked      0
Title         0
FamilySize    0
IsAlone       0
dtype: int64

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone
0,0,3,male,22.0,1,0,7.2500,S,Mr,2,0
1,1,1,female,38.0,1,0,71.2833,C,Mrs,2,0
2,1,3,female,26.0,0,0,7.9250,S,Miss,1,1
3,1,1,female,35.0,1,0,53.1000,S,Mrs,2,0
4,0,3,male,35.0,0,0,8.0500,S,Mr,1,1


## Part 2: Model

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# Separate the target from the features
X = df.drop('Survived', axis=1)
y = df['Survived']

# Convert categorical columns to numbers for sklearn
X = pd.get_dummies(X, drop_first=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=20)

# Helper function for evaluation
def evaluate_model(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print(model_name)
    print('Accuracy:', acc)
    print('Precision:', prec)
    print('Recall:', rec)
    print('Confusion Matrix:')
    print(pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1']))
    print()

    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec}

### Logistic Regression

In [8]:
# Train and evaluate the logistic regression model
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_result = evaluate_model('Logistic Regression', y_test, lr_pred)

Logistic Regression
Accuracy: 0.8171641791044776
Precision: 0.7525773195876289
Recall: 0.7448979591836735
Confusion Matrix:
          Predicted 0  Predicted 1
Actual 0          146           24
Actual 1           25           73



### Decision Tree

In [9]:
# Train and evaluate the decision tree model
dt = DecisionTreeClassifier(random_state=20)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_result = evaluate_model('Decision Tree', y_test, dt_pred)

# Compare the models
results = pd.DataFrame([lr_result, dt_result]).sort_values('Accuracy', ascending=False).reset_index(drop=True)
results

Decision Tree
Accuracy: 0.7873134328358209
Precision: 0.6990291262135923
Recall: 0.7346938775510204
Confusion Matrix:
          Predicted 0  Predicted 1
Actual 0          139           31
Actual 1           26           72



,Model,Accuracy,Precision,Recall
0,Logistic Regression,0.817164,0.752577,0.744898
1,Decision Tree,0.787313,0.699029,0.734694


### Suggestions to Improve the Models

Logistic Regression performed slightly better than the Decision Tree because it had higher accuracy and precision on this dataset.

To improve the models, I would try scaling the numeric columns before Logistic Regression and limiting the depth of the Decision Tree so it does not overfit.

I could also test a few extra engineered features, such as whether a passenger has a cabin record or a surname-based family group.